<a href="https://colab.research.google.com/github/Autosnight/COMP90042_2026/blob/andrew-branch/embedding_to_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#installing libraries
%pip install transformers

In [3]:
# 1. Update core libraries and resolve torchao dependency conflict
%pip install --upgrade transformers sentence-transformers torchao

print("\n--- SETUP COMPLETE ---")
print("IMPORTANT: You MUST restart the session now (Runtime > Restart session) before running the model loading cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.1
    Uninstalling sentence-transformers-5.4.1:
      Successfully uninstalled sentence-transformers-5.4.1

--- SETUP COMPLETE ---
IMPORTANT: You MUST restart the session now (Runtime > Restart session) before running the model loading cell.


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression #using log reg due to quick training and inference, good baseline measure
import torch
import transformers as ppb
import warnings
warnings.filterwarnings('ignore')

# Loading data
/content/evidence.json
and /content/COMP90042_2026/data/train-claims.json
convert claims into table with 2 colums: claim text and labels
convert evidence into list of strings

In [4]:
import pandas as pd
import json

# Load claims data directly using pandas
claims_df = pd.read_json('COMP90042_2026/data/train-claims.json', orient='index')

# Prepare claims_df columns
claims_df = claims_df[['claim_text', 'claim_label']]
claims_df.rename(columns={'claim_label': 'label'}, inplace=True)

# Load evidence data from root directory
with open('evidence.json', 'r') as f:
    evidence_data = json.load(f)

evidence_series = pd.Series(evidence_data)
evidence_list = evidence_series.tolist()

display(claims_df.head())
print(f"Loaded {len(claims_df)} claims and {len(evidence_list)} evidence items.")

FileNotFoundError: File COMP90042_2026/data/train-claims.json does not exist

# Loading frozen models + embeddings
1 cell: load distilbert and jina v5. jina v5 retrieval nano model.
different cell: use sentence transformer to convert all claims and evidence into jina embeddings (jina.encode). claims should be query embeddings, evidence should be document embeddings


In [4]:
from sentence_transformers import SentenceTransformer
import torch

# Load Jina v5 nano model
# trust_remote_code=True is required for Jina v5's custom architecture
jina_model = SentenceTransformer(
    "jinaai/jina-embeddings-v5-text-nano",
    trust_remote_code=True
)

print("Jina v5 Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modeling_eurobert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-nano:
- modeling_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-nano:
- modeling_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/424M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/582 [00:00<?, ?B/s]

Jina v5 Model loaded successfully!


### Loading Jina v5 with Custom Config
If you have just restarted your session, run the cell below to load the model. The `trust_remote_code=True` parameter is essential for downloading the custom `eurobert` architecture.

In [9]:
# Generate embeddings using Jina v5
print("Encoding claims as queries...")

# Ensure claims_df and evidence_list exist in current session
# If you get a NameError, please execute cell 513ac303 first.
claim_texts = claims_df['claim_text'].tolist()

claim_embeddings = jina_model.encode(
    sentences=claim_texts,
    task="retrieval",
    prompt_name="query"
)

print("Encoding a subset of evidence as documents...")
# Use first 1000 items for efficiency and initial testing
subset_evidence = evidence_list[:1000]

evidence_embeddings = jina_model.encode(
    sentences=subset_evidence,
    task="retrieval",
    prompt_name="document"
)

print(f"Generated {claim_embeddings.shape[0]} claim embeddings and {evidence_embeddings.shape[0]} evidence embeddings.")

Encoding claims as queries...


`use_return_dict` is deprecated! Use `return_dict` instead!


Encoding a subset of evidence as documents...
Generated 1228 claim embeddings and 1000 evidence embeddings.


In [10]:
#retrieve class and weights
model_class, pretrained_weights = (ppb.DistilBertModel, 'distilbert-base-uncased')
#load model
frozen_bert = model_class.from_pretrained(pretrained_weights)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# retrieval
use jina similarity function with jina embeddings to get 2 most similar texts from documents/evidence for each claim. add them to a dictionary with key = claim (string) and value = evidences (string)

In [15]:
import numpy as np
import json # Added for loading json files
import pandas as pd # Added for DataFrame operations
from sentence_transformers import SentenceTransformer # Added for loading Jina model

claim_evidence_map = {}

# --- Robustness checks for dependencies ---

# Check and load evidence_list if not present
if 'evidence_list' not in locals() and 'evidence_list' not in globals():
    print("Warning: 'evidence_list' was not found. Attempting to load from 'evidence.json'.")
    try:
        with open('evidence.json', 'r') as f:
            evidence_data = json.load(f)
        evidence_list = list(evidence_data.values())
        print(f"Successfully loaded {len(evidence_list)} evidence items.")
    except FileNotFoundError:
        raise FileNotFoundError("Error: 'evidence.json' not found. Please ensure data files are in place.")
    except Exception as e:
        raise Exception(f"Error loading 'evidence.json': {e}")

# Check and load jina_model if not present
if 'jina_model' not in locals() and 'jina_model' not in globals():
    print("Warning: 'jina_model' was not found. Attempting to load 'jinaai/jina-embeddings-v5-text-nano'.")
    try:
        jina_model = SentenceTransformer(
            "jinaai/jina-embeddings-v5-text-nano",
            trust_remote_code=True
        )
        print("Successfully loaded 'jina_model'.")
    except Exception as e:
        raise Exception(f"Error loading 'jina_model': {e}. Please ensure cell b6f8a5d3 has been run and session restarted if necessary.")

# Check and load claims_df if not present (needed for claim_texts)
if 'claims_df' not in locals() and 'claims_df' not in globals():
    print("Warning: 'claims_df' was not found. Attempting to load from 'COMP90042_2026/data/train-claims.json'.")
    try:
        with open('COMP90042_2026/data/train-claims.json', 'r') as f:
            claims_data = json.load(f)
        claims_df = pd.DataFrame.from_dict(claims_data, orient='index')
        claims_df = claims_df[['claim_text', 'claim_label']]
        claims_df.rename(columns={'claim_label': 'label'}, inplace=True)
        print(f"Successfully loaded {len(claims_df)} claims.")
    except FileNotFoundError:
        raise FileNotFoundError("Error: 'COMP90042_2026/data/train-claims.json' not found. Please ensure data files are in place.")
    except Exception as e:
        raise Exception(f"Error loading 'claims_df': {e}")

# Check and define claim_texts if not present
if 'claim_texts' not in locals() and 'claim_texts' not in globals():
    print("Warning: 'claim_texts' was not found. Generating from 'claims_df'.")
    claim_texts = claims_df['claim_text'].tolist()
    print(f"Successfully generated {len(claim_texts)} claim texts.")

# Check and encode claim_embeddings if not present
if 'claim_embeddings' not in locals() and 'claim_embeddings' not in globals():
    print("Warning: 'claim_embeddings' was not found. Encoding claims as queries.")
    claim_embeddings = jina_model.encode(
        sentences=claim_texts,
        task="retrieval",
        prompt_name="query"
    )
    print("Successfully encoded 'claim_embeddings'.")

# Check and define subset_evidence_list if not present
if 'subset_evidence_list' not in locals() and 'subset_evidence_list' not in globals():
    print("Warning: 'subset_evidence_list' was not found. Creating it from 'evidence_list'.")
    subset_evidence_list = evidence_list[:1000] # Ensure consistency with previous truncation
    print(f"Successfully created 'subset_evidence_list' with {len(subset_evidence_list)} items.")

# Check and encode evidence_embeddings if not present
if 'evidence_embeddings' not in locals() and 'evidence_embeddings' not in globals():
    print("Warning: 'evidence_embeddings' was not found. Encoding evidence as documents.")
    evidence_embeddings = jina_model.encode(
        sentences=subset_evidence_list,
        task="retrieval",
        prompt_name="document"
    )
    print("Successfully encoded 'evidence_embeddings'.")

# Calculate similarity using jina_model's similarity function
# jina_model.similarity returns a matrix of cosine similarities
similarities = jina_model.similarity(claim_embeddings, evidence_embeddings)

# Convert to numpy if it's a tensor for easier indexing
if hasattr(similarities, 'numpy'):
    similarities = similarities.numpy()

for i, claim in enumerate(claim_texts):
    # Get indices of top 2 highest similarity scores
    top_2_indices = np.argsort(similarities[i])[-2:][::-1]
    retrieved_evidence = [evidence_list[idx] for idx in top_2_indices]
    claim_evidence_map[claim] = retrieved_evidence

print(f"Mapped {len(claim_evidence_map)} claims to their top 2 evidence pieces using Jina similarity.")

Successfully created 'subset_evidence_list' with 1000 items.
Mapped 1228 claims to their top 2 evidence pieces using Jina similarity.


# concatination:
concatinate these strings together and add sep tokens between and then encode them for distilbert with padding  

In [2]:
tokenizer = ppb.DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

concatenated_inputs = []
for claim, evidences in claim_evidence_map.items():
    # Concatenate: [CLS] Claim [SEP] Evidence1 [SEP] Evidence2 [SEP]
    text = f"{claim} [SEP] {evidences[0]} [SEP] {evidences[1]}"
    concatenated_inputs.append(text)

tokenized_features = tokenizer(
    concatenated_inputs,
    padding='max_length',
    truncation=True,
    max_length=512,
    return_tensors='pt'
)

NameError: name 'ppb' is not defined

# classifier embeddings
use distilbert to create cls embeddings of concatinated claim-evidence-evidence strings

In [1]:
frozen_bert.eval()
all_features = []

with torch.no_grad():
    # Processing in small batches to avoid OOM
    batch_size = 8
    for i in range(0, len(concatenated_inputs), batch_size):
        batch_ids = tokenized_features['input_ids'][i:i+batch_size]
        batch_mask = tokenized_features['attention_mask'][i:i+batch_size]

        outputs = frozen_bert(batch_ids, attention_mask=batch_mask)
        # Extract [CLS] token embeddings (first token)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_features.append(cls_embeddings)

features = np.concatenate(all_features, axis=0)
labels = claims_df['label'].values
print(f"Feature matrix shape: {features.shape}")

NameError: name 'frozen_bert' is not defined

#

# Training classification model

---
## input:

features and labels,

Output: accuracy scores in train and test set


In [ ]:
# Splitting the data
train_features, test_features, train_labels, test_labels = train_test_split(features, labels)

# Initialize and train the Logistic Regression model
completed_classifier = LogisticRegression(max_iter=1000)
completed_classifier.fit(train_features, train_labels)

# Calculate accuracy scores
train_score = completed_classifier.score(train_features, train_labels)
test_score = completed_classifier.score(test_features, test_labels)

print(f"Training Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")

In [ ]:
import json

# Get predictions from the trained classifier
predictions = completed_classifier.predict(features)

# We need the original evidence IDs for the output format
evidence_keys = list(evidence_data.keys())
output_results = {}

# Iterate through the original claims_data to maintain ID consistency
for i, (claim_id, claim_info) in enumerate(claims_data.items()):
    # Re-retrieve the top 2 indices for this specific claim to get the evidence IDs
    top_2_indices = np.argsort(similarities[i])[-2:][::-1]
    retrieved_evidence_ids = [evidence_keys[idx] for idx in top_2_indices]

    output_results[claim_id] = {
        "claim_text": claim_info['claim_text'],
        "claim_label": predictions[i],
        "evidences": retrieved_evidence_ids
    }

# Save to a JSON file
with open('predictions.json', 'w') as f:
    json.dump(output_results, f, indent=2)

print("Results formatted and saved to predictions.json")
# Display a sample of the formatted output
print(json.dumps(dict(list(output_results.items())[:2]), indent=2))